In [ ]:
import lsdb

lsdb.__version__

In [ ]:
# Dask puts out more advisory logging than we care for in this tutorial.
# It takes some doing to quiet all of it, but this recipe works.

import dask

dask.config.set({"logging.distributed": "critical"})

import logging

# This also has to be done, for the above to be effective
logger = logging.getLogger("distributed")
logger.setLevel(logging.CRITICAL)

import warnings

# Finally, suppress the specific warning about Dask dashboard port usage
warnings.filterwarnings("ignore", message="Port 8787 is already in use.")

In [ ]:
from dask.distributed import Client

client = Client(n_workers=3, threads_per_worker=1, memory_limit="4g")
client


Your Dask dashboard will be accessible at https://rsp.lsst.ac.uk/nb/user/davedavemckay/proxy/8787/status

Set Dask S3 endpoint

### 1.3 Opening a Catalog

The data is divided into `objects` and `dia_objects`.  Let's open both catalogs:

In [ ]:
from upath import UPath

base_path = UPath("/rubin/lsdb_data")

object_cat = lsdb.open_catalog(base_path / "object_collection", columns='all')
# dia_object_cat = lsdb.open_catalog(base_path / "dia_object_collection")

In [ ]:
object_cat

In [ ]:
import re
search_string = re.compile('Flux')
band = re.compile('i_')
exclude1 = re.compile('Err')
exclude2 = re.compile('flag')
# l = ['a','b','Flux_is_good']
# a = [ w if search_string.search(w) else 'nope' for w in l ]
# a
cols = [ c for c in object_cat.columns if search_string.search(c) and band.search(c) and not exclude1.search(c) and not exclude2.search(c) ]
cols

In [ ]:
search_string = re.compile('Mag')
band = re.compile('i_')
exclude1 = re.compile('Err')
cols = [ c for c in object_cat.columns if search_string.search(c) and band.search(c) and not exclude1.search(c) ]
cols

In [ ]:
target_ra = 53.2
target_dec = -28.1

nanoJanskyToABMag:

$m = -2.5 log(f) + 31.4$

where _m_ is _magnitude_ and _f_ is _flux_.

In [ ]:
import math
def nanoJanskyToABMag(flux):
    return -2.5 * math.log(flux) + 31.4

In [ ]:
from astropy.coordinates import SkyCoord
import astropy.units as u

def cone_search(source_ra, source_dec):
    center = SkyCoord(ra=source_ra*u.deg, dec=source_dec*u.deg, frame='icrs')
    target = SkyCoord(ra=target_ra*u.deg, dec=target_dec*u.deg, frame='icrs')
    radius = 0.5 * u.deg
    sep = center.separation(target)
    return sep <= radius

In [ ]:
galaxies = object_cat.cone_search(target_ra, target_dec, radius_arcsec=0.5)

In [ ]:
galaxies

In [ ]:
object_cat = object_cat.
# cmf = object_cat[['i_cModelFlux']].copy()
# cmf

In [ ]:
galaxies = galaxies[nanoJanskyToABMag(galaxies['i_cModelFlux']) > 20]

In [ ]:
galaxies = object_cat.where(
    object_cat['i_cModelFlux'] / object_cat['i_cModelFluxErr'] > 20 and
    object_cat['i_extendedness'] == 1 and
    object_cat['sersic_no_data_flag'] == 0 and
    object_cat['i_kronFlux_flag'] == 0 and
    object_cat['i_cModel_flag'] == 0 and
    nanoJanskyToABMag(object_cat['i_cModelFlux']) > 20 and
    cone_search(object_cat['coord_ra'], object_cat['coord_dec'])[[
        'objectId',
        'coord_ra',
        'coord_dec',
        'detect_fromBlend',
        'detect_isIsolated',
        'i_blendedness',
        'i_extendedness',
        'i_kronFlux',
        'i_kronFluxErr',
        'i_kronRad',
        'i_cModelFlux',
        'i_cModelFluxErr',
        'i_gaap1p0Flux',
        'g_gaap1p0Flux',
        'g_gaap3p0Flux',
        'sersic_index',
        'i_sersicFlux',
        'i_bdFluxB',
        'i_bdFluxD',
        'i_kronFlux_flag',
        'i_cModel_flag',
        'sersic_no_data_flag',
        'shape_xx',
        'shape_xy',
        'shape_yy',
        'i_ap03Flux'
        'i_ap06Flux',
        'i_ap09Flux',
        'i_ap12Flux',
        'i_ap17Flux',
        'i_ap25Flux',
        'i_ap35Flux',
        'i_ap50Flux',
    ]]
).copy()

In [ ]:
query = "SELECT obj.objectId, obj.coord_ra, obj.coord_dec, " + \
        "obj.detect_fromBlend, obj.detect_isIsolated, " + \
        "obj.i_blendedness, obj.i_extendedness, " + \
        "obj.i_kronFlux, obj.i_kronFluxErr, obj.i_kronRad, " + \
        "obj.i_cModelFlux, obj.i_cModelFluxErr, obj.i_gaap1p0Flux, " + \
        "obj.g_gaap1p0Flux, obj.g_gaap3p0Flux, obj.sersic_index, " + \
        "obj.i_sersicFlux, obj.i_bdFluxB, obj.i_bdFluxD, " + \
        "obj.i_kronFlux_flag, obj.i_cModel_flag, obj.sersic_no_data_flag, " + \
        "obj.shape_xx, obj.shape_xy, obj.shape_yy, " + \
        "obj.i_ap03Flux, obj.i_ap06Flux, obj.i_ap09Flux, obj.i_ap12Flux, " + \
        "obj.i_ap17Flux, obj.i_ap25Flux, obj.i_ap35Flux, obj.i_ap50Flux " + \
        "FROM dp1.Object AS obj " + \
        "WHERE (obj.i_cModelFlux/obj.i_cModelFluxErr > 20) AND " + \
        "(obj.i_extendedness = 1) AND (obj.sersic_no_data_flag = 0) AND " + \
        "(obj.i_kronFlux_flag = 0) AND (obj.i_cModel_flag = 0) AND " + \
        "(scisql_nanojanskyToAbMag(obj.i_cModelFlux) > 20) AND " + \
        "CONTAINS(POINT('ICRS', obj.coord_ra, obj.coord_dec), " + \
        "CIRCLE('ICRS',"+str(target_ra)+","+str(target_dec)+", 0.1)) = 1 "

In [ ]:
dia_object_cat

In [ ]:
dia_object_cat.all_columns

In [ ]:
query = "SELECT mpc.ssObjectId, "\
        "mpc.q, mpc.e, mpc.incl, "\
        "sso.discoverySubmissionDate "\
        "FROM dp1.MPCORB as mpc "\
        "INNER JOIN dp1.SSObject as sso "\
        "ON mpc.ssObjectId = sso.ssObjectId"

In [1]:
import s3fs
with open('../.aws/credentials', 'r') as credf:
    creds = {
        l.split(' = ')[0].strip():l.split(' = ')[1].strip() for l in credf.readlines() if l.startswith('aws')
    }
creds['endpoint_url'] = 'https://somerville.ed.ac.uk:6780'
tarcs_s3 = s3fs.S3FileSystem(
      key=creds['aws_access_key_id'],
      secret=creds['aws_secret_access_key'],
      endpoint_url=creds['endpoint_url']
   )

In [2]:
tarcs_s3.ls('tarcs')

['tarcs/dask_scheduler_info.txt', 'tarcs/test']

In [5]:
tarcs_s3.get('tarcs/dask_scheduler_info.txt','dask_scheduler_info.txt')

[None]

In [ ]:
remote_path = UPath("s3://tarcs/ob_c")

In [ ]:
object_cat.write_catalog(remote_path, overwrite=True)

In [ ]:
# tarcs_s3.put('/rubin/lsdb_data/object_collection/', 'tarcs/object_collection', recursive=True, batch_size=10)

In [ ]:
# tarcs_s3.ls('tarcs/object_collection/object_lc_5arcs/dataset/Norder=9/Dir=1320000/Npix=1324370.parquet')